# RAG exemplo2
## documento pdf anuario de segurança publica


## Libraries

In [1]:
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')
import datetime
import time
import requests
from tqdm.auto import tqdm



# # Modelos LLM (Large Language Models)
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.language_models.chat_models import BaseChatModel

#embedding
from langchain_huggingface import HuggingFaceEmbeddings  



from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    MessagesPlaceholder,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate
)

from langchain_core.output_parsers import StrOutputParser


# Criação e execução de agentes
from langchain_classic.agents import( 
Tool, 
AgentExecutor,
create_tool_calling_agent,
create_react_agent)

# # Ferramentas customizadas para agentes
from langchain.tools import tool
from langchain_community.agent_toolkits.load_tools import load_tools
from langchain_experimental.tools.python.tool import PythonAstREPLTool

from langchain_classic.memory import ConversationBufferMemory


from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter,MarkdownHeaderTextSplitter

# # Componentes de RAG (Retrieval-Augmented Generation)
from langchain_chroma import Chroma  # Armazenamento vetorial


print('#',time.strftime("%d/%m/%Y - %H:%M:%S"))



# 12/08/2026 - 08:18:04


In [2]:
# 1. Configuração do diretório de saída
OUTPUT_DOCUMENTS_DIR: str = './anuario/'
os.makedirs(OUTPUT_DOCUMENTS_DIR, exist_ok=True)

# 2. Carregamento das variáveis de ambiente (.env)
ENV_PATH: str = '/home/akel/PycharmProjects/InsurMinds2026/.env'

def carrega_variaveis_ambiente() -> None:
    if os.path.exists(ENV_PATH):
        load_dotenv(ENV_PATH, override=True)
        print("✔ Variáveis de ambiente carregadas do arquivo .env")
    else:
        print(f"⚠ Aviso: Arquivo {ENV_PATH} não foi encontrado no diretório atual.")

print('✔ OUTPUT_DOCUMENTS_DIR:',OUTPUT_DOCUMENTS_DIR)
carrega_variaveis_ambiente()


llm_gemini = ChatGoogleGenerativeAI(temperature=0, model="gemini-3.1-flash-lite-preview",google_api_key=os.getenv("GOOGLE_API"))

print("✔ LLM carregada:",llm_gemini.profile['name'])


✔ OUTPUT_DOCUMENTS_DIR: ./anuario/
✔ Variáveis de ambiente carregadas do arquivo .env
✔ LLM carregada: Gemini 3.1 Flash Lite Preview


## Criando Banco de dados
### 1.leitura da fonte

In [3]:
# acessando documento 
document = PyPDFDirectoryLoader(OUTPUT_DOCUMENTS_DIR).load()
print("✔ documento carregado")
print(" Total de paginas:", document[0].metadata[ 'total_pages'])
print('#',time.strftime("%d/%m/%Y - %H:%M:%S"))


✔ documento carregado
 Total de paginas: 424
# 11/08/2026 - 16:56:22


### 2.Splitter

In [4]:

#Splitter document
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,        # ← menor: mais preciso na recuperação
    chunk_overlap=150,     # ← overlap proporcional
    separators=["\n\n", "\n", ".", "!", "?", " "],  # ← respeita parágrafos
    length_function=len,
)

split_documents = text_splitter.split_documents(document)

for i, split in enumerate(split_documents):
    split.metadata.update({
        "chunk_id": i,
        "chunk_total": len(split_documents),
        "posicao": f"{i/len(split_documents )*100:.0f}%"  # posição no livro
    })
print("# Documento fatiado em:", time.strftime("%d/%m/%Y - %H:%M:%S"))
print(" chunk_total:", len(split_documents ))
print("--------------------")



# Documento fatiado em: 11/08/2026 - 16:56:27
 chunk_total: 1118
--------------------


### 3.Embedding Model

In [16]:
#embedding
model_embedding1='intfloat/multilingual-e5-small'
model_embedding2='intfloat/multilingual-e5-large-instruct'
embedding_e5= HuggingFaceEmbeddings(
    model_name=model_embedding2,
    model_kwargs={
        "device": "cpu",
        "trust_remote_code": True},
    encode_kwargs={
        "normalize_embeddings": True,
        "prompt": "passage: " })

print('#',time.strftime("%d/%m/%Y - %H:%M:%S"))


KeyboardInterrupt



### 4.VectorStore

In [ ]:
#vectorstore = Chroma.from_documents(chunks, embedding_e5, persist_directory=f'{OUTPUT_DOCUMENTS_DIR}vectorstore')


In [6]:
# # Tamanho de cada lote
batch_size = 150
total = len(split_documents )

# Cria o banco com o primeiro lote
print(f"\n#Iniciando criação do banco: {total} chunks")
print(f"#Início: {time.strftime('%H:%M:%S')}\n")

# vectorstore_small
vectorstore = Chroma.from_documents(
    documents=split_documents[:batch_size],
    embedding=embedding_e5,
    persist_directory=f'{OUTPUT_DOCUMENTS_DIR}vectorstore_large',
    collection_metadata={"hnsw:space": "cosine"}
)

# # Adiciona os lotes restantes com progresso
for i in tqdm(range(batch_size, total, batch_size), desc="Indexando chunks"):
    lote = split_documents[i:i + batch_size]
    vectorstore.add_documents(lote)
    print(f"  Lote {i//batch_size + 1}/{total//batch_size} | "
          f"Chunks {i}–{min(i+batch_size, total)} | "
          f"{time.strftime('%H:%M:%S')}")

print(f"\n#Banco criado com {vectorstore._collection.count()} chunks.")
print(f"#Finalizado em: {time.strftime('%H:%M:%S')}")


#Iniciando criação do banco: 1118 chunks
#Início: 16:57:35



Indexando chunks:  14%|██████████████████▊                                                                                                                 | 1/7 [26:06<2:36:39, 1566.55s/it]

  Lote 2/7 | Chunks 150–300 | 17:49:53


Indexando chunks:  29%|█████████████████████████████████████▋                                                                                              | 2/7 [52:56<2:12:41, 1592.33s/it]

  Lote 3/7 | Chunks 300–450 | 18:16:43


...

  Lote 6/7 | Chunks 750–900 | 19:35:22


Indexando chunks:  86%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏                  | 6/7 [2:38:28<26:35, 1595.32s/it]

  Lote 7/7 | Chunks 900–1050 | 20:02:14


Indexando chunks: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [2:48:59<00:00, 1448.44s/it]

  Lote 8/7 | Chunks 1050–1118 | 20:12:45

#Banco criado com 1118 chunks.
#Finalizado em: 20:12:45


## Carregando o banco vetorial criado

In [3]:
def carrega_banco_de_dados_vetorial(path_documentos:str) -> Chroma:
    try:
        model_embedding1='intfloat/multilingual-e5-small'
        model_embedding2='intfloat/multilingual-e5-large-instruct'
        embedding_query = HuggingFaceEmbeddings(
            model_name=model_embedding2,
            model_kwargs={"device": "cpu",
                          "trust_remote_code": True},
            encode_kwargs={
                "normalize_embeddings": True,
                "prompt": "query: "   })

        vectorstore = Chroma(persist_directory=path_documentos, embedding_function=embedding_query)
        if vectorstore:
            print('banco de dados carregado!')
        
        return vectorstore
    except Exception as e:
        print(f"Erro ao carregar o banco de dados vetorial: {e}")
        return None

In [4]:
vectorstore = carrega_banco_de_dados_vetorial(f'{OUTPUT_DOCUMENTS_DIR}vectorstore_large')
# docs = None
# retriever = vectorstore.as_retriever()
# docs = retriever.invoke("Data H")


Loading weights: 100%|█████████████████████| 391/391 [00:00<00:00, 1285.73it/s]


banco de dados carregado!


## criando contexto com a base de dados

In [9]:
def busca_na_base_de_documentos(pergunta:str) -> str:
    """Use esta ferramenta para responder perguntas sobre o anuario de segurança publica."""
    vectorstore = carrega_banco_de_dados_vetorial(f'{OUTPUT_DOCUMENTS_DIR}vectorstore_large')
    contexto = None
    if vectorstore:
        retriever = vectorstore.as_retriever()
        docs = retriever.invoke(pergunta)
        contexto = "\n\n".join([doc.page_content for doc in docs])
    return contexto


## Criando agente RAG

In [10]:
def string_gemini(out_agent_exe):
    if isinstance(out_agent_exe, list) and len(out_agent_exe) > 0:
        if isinstance(out_agent_exe[0], dict) and 'text' in out_agent_exe[0]:
            return out_agent_exe[0]['text']
    
    return  out_agent_exe[0]['text']


def agente_langchain_RAG1(modelo_llm=llm_gemini) -> dict:
    ferramentas = []
    memoria = ConversationBufferMemory(memory_key="chat_history", return_messages=True, input_key="input")

    prompt = PromptTemplate(
        input_variables=["input", "context", "chat_history", "agent_scratchpad"],
        template=""" {chat_history}
                Você é um agente de IA especializado em responder perguntas de segurança publica. Responda apenas informações que estão 
                dentro do seu contexto.Jamais busque informações de outras fontes
                Contexto: {context}
                Pergunta: {input}
                {agent_scratchpad}
        """)

    agente = create_tool_calling_agent(modelo_llm,ferramentas, prompt)
    executor_do_agente = AgentExecutor(agent=agente, tools=ferramentas, memory=memoria)
    return executor_do_agente

#### Executando sem contexto

In [11]:
executor_do_agente = agente_langchain_RAG1()

pergunta1 = "comente sobre taxa de Mortes Violentas Intencionais 2024 e 2025"
contexto1 = ''
resposta1 = executor_do_agente.invoke({"input": pergunta1, "context": contexto1})
resposta1=string_gemini(resposta1['output'])

print(resposta1)
print('\n' + '=' * 10)

O contexto fornecido está vazio, portanto, não há informações disponíveis para responder à sua pergunta sobre as taxas de Mortes Violentas Intencionais de 2024 e 2025.



In [12]:
executor_do_agente = agente_langchain_RAG1()

pergunta1 = "Comente Roubos e Furtos de celular nas capitais.Números e evolução."
contexto = busca_na_base_de_documentos(pergunta1)
resposta2 = executor_do_agente.invoke({"input": pergunta1, "context": contexto})
resposta2=string_gemini(resposta2['output'])
#
print(resposta2)
print('\n' + '=' * 10)

Loading weights: 100%|█████████████████████| 391/391 [00:00<00:00, 6697.18it/s]


banco de dados carregado!
Com base nos dados fornecidos, aqui estão as informações sobre roubos e furtos de celulares nas capitais e o cenário nacional:

**Concentração nas Capitais e Regiões Metropolitanas**
Os dados de 2025 revelam que as maiores taxas de roubos e furtos de celular estão fortemente concentradas em grandes centros urbanos. Entre os 20 municípios com as maiores taxas do país, 14 são capitais estaduais. Os demais municípios que compõem esse grupo são cidades vizinhas a grandes capitais, integradas às suas dinâmicas metropolitanas.

**Evolução e Tendência Nacional**
Embora o contexto destaque a concentração nas capitais, o cenário nacional em 2025 apresentou uma tendência de queda no indicador agregado de roubos e furtos de celulares, com redução observada em 25 das 27 Unidades da Federação (UFs).

*   **Reduções mais expressivas:** Rio Grande do Norte (-22,1%), Mato Grosso (-20,6%), Goiás (-19,6%), Amazonas (-19,2%) e Espírito Santo (-18,2%).
*   **Elevações:** Apenas d

In [13]:
pergunta2 = "Quais capitais obteve destaque?"
contexto = busca_na_base_de_documentos(pergunta2)
resposta3 = executor_do_agente.invoke({"input": pergunta2, "context": contexto})
resposta3=string_gemini(resposta3['output'])
print(resposta3)
print('\n' + '=' * 10)

Loading weights: 100%|█████████████████████| 391/391 [00:00<00:00, 7005.20it/s]


banco de dados carregado!
Com base no contexto fornecido, não há informações sobre quais capitais obtiveram destaque em relação a roubos e furtos de celular. O texto disponibilizado menciona apenas que a Bahia conta com cinco cidades no ranking e o Ceará com três, sem especificar se são capitais ou quais seriam as demais cidades.



In [14]:
pergunta3 = "há informações  onde os lares são mais violentos para crianças"
contexto1 = busca_na_base_de_documentos(pergunta3)
resposta3 = executor_do_agente.invoke({"input": pergunta3, "context": contexto1})
resposta3=string_gemini(resposta3['output'])
print(resposta3)
print('\n' + '=' * 10)

Loading weights: 100%|█████████████████████| 391/391 [00:00<00:00, 7405.51it/s]


banco de dados carregado!
Com base no contexto fornecido, não há informações específicas que identifiquem geograficamente onde os lares são mais violentos para crianças.

O texto limita-se a informar que a violência contra crianças e adolescentes é, em sua maioria, **intrafamiliar** e que a frequência escolar é um fator fundamental para a denúncia desses casos, uma vez que a ausência de convívio escolar (como observado no período de lockdown) leva ao represamento das denúncias.



In [15]:
pergunta = "há informações  onde os lares são mais violentos para mulheres"
contexto = busca_na_base_de_documentos(pergunta)
resposta = executor_do_agente.invoke({"input": pergunta, "context": contexto})
resposta=string_gemini(resposta['output'])
print(resposta)
print('\n' + '=' * 10)

Loading weights: 100%|█████████████████████| 391/391 [00:00<00:00, 6257.31it/s]


banco de dados carregado!
Com base no contexto fornecido, não há informações geográficas ou específicas que identifiquem onde (quais locais ou regiões) os lares são mais violentos para as mulheres.

O documento menciona apenas o dado estatístico de que **56,5% das vítimas de feminicídio foram mortas na residência**.



In [22]:
pergunta = "comente sobre os estados com maiores indicies de violencia contra mulher"
contexto = busca_na_base_de_documentos(pergunta)
resposta = executor_do_agente.invoke({"input": pergunta, "context": contexto})
resposta=string_gemini(resposta['output'])
print(resposta)
print('\n' + '=' * 10)

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 5656.42it/s]


banco de dados carregado!
Com base no contexto fornecido, a violência contra a mulher apresenta uma heterogeneidade territorial significativa, com destaque para os seguintes pontos:

*   **Feminicídios:**
    *   **Maiores taxas:** O Acre lidera com a maior taxa do país (3,2 vítimas por 100 mil mulheres), seguido por Rondônia (2,9).
    *   **Crescimento:** O Amapá apresentou o maior crescimento na taxa de feminicídios (347,7%), alcançando 2,2 vítimas por 100 mil mulheres.
    *   **Proporção de registros:** Em estados como Acre, Distrito Federal e Sergipe, a proporção de homicídios de mulheres classificados como feminicídio aproxima-se de 70%. Em contraste, o Ceará apresenta a maior taxa de homicídios femininos (6,2), mas uma das menores taxas de feminicídio (1,0), o que sugere uma prática institucional de classificar a maioria desses casos como homicídios comuns.

*   **Estupros:**
    *   **Maiores taxas (população geral):** O texto menciona que os estados com as maiores taxas de es